# 🚀 Chapter 09-01b: Hands-On Fine-Tuning with OpenAI API

---

## 📖 The Mission

Fine-tune GPT-4o-mini to be a specific assistant — one that always responds in a particular style and format. We'll walk through the complete OpenAI fine-tuning pipeline:

1. Prepare training data
2. Validate the dataset
3. Upload to OpenAI
4. Create fine-tuning job
5. Use the fine-tuned model

---

## 🧪 Step 1: Setup

In [ ]:
# !pip install openai python-dotenv

In [ ]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print("✅ OpenAI client ready!")

## 🧪 Step 2: Prepare Training Data (JSONL Format)

OpenAI requires chat-format JSONL: each line is a conversation with `system`, `user`, `assistant` messages.

In [ ]:
# Create sample training data — a helpful coding tutor that always:
# 1. Explains the concept first
# 2. Shows code example
# 3. Gives a practice exercise

training_data = [
    {
        "messages": [
            {"role": "system", "content": "You are a coding tutor. Always: 1) Explain the concept, 2) Show a code example, 3) Give a practice exercise."},
            {"role": "user", "content": "What is a Python list comprehension?"},
            {"role": "assistant", "content": "**Concept:** A list comprehension is a concise way to create lists in Python. It combines a for loop and optional condition into a single line.\n\n**Example:**\n```python\n# Traditional\nsquares = []\nfor x in range(5):\n    squares.append(x**2)\n\n# List comprehension\nsquares = [x**2 for x in range(5)]\nprint(squares)  # [0, 1, 4, 9, 16]\n```\n\n**Practice:** Create a list comprehension that filters even numbers from 1-20 and doubles them."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a coding tutor. Always: 1) Explain the concept, 2) Show a code example, 3) Give a practice exercise."},
            {"role": "user", "content": "Explain Python decorators."},
            {"role": "assistant", "content": "**Concept:** A decorator is a function that wraps another function to extend its behavior without modifying it. Think of it as gift wrapping — the gift (function) stays the same, but you add a bow (extra behavior).\n\n**Example:**\n```python\ndef timer(func):\n    import time\n    def wrapper(*args, **kwargs):\n        start = time.time()\n        result = func(*args, **kwargs)\n        print(f'{func.__name__} took {time.time()-start:.2f}s')\n        return result\n    return wrapper\n\n@timer\ndef slow_function():\n    time.sleep(1)\n    return 'done'\n```\n\n**Practice:** Write a decorator called `@retry` that retries a function up to 3 times if it raises an exception."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a coding tutor. Always: 1) Explain the concept, 2) Show a code example, 3) Give a practice exercise."},
            {"role": "user", "content": "What are Python generators?"},
            {"role": "assistant", "content": "**Concept:** Generators are functions that yield values one at a time, creating an iterator lazily. Unlike lists that store everything in memory, generators produce values on-demand — perfect for large datasets.\n\n**Example:**\n```python\ndef fibonacci(n):\n    a, b = 0, 1\n    for _ in range(n):\n        yield a\n        a, b = b, a + b\n\n# Only generates values as needed\nfor num in fibonacci(10):\n    print(num, end=' ')\n# 0 1 1 2 3 5 8 13 21 34\n```\n\n**Practice:** Write a generator that yields prime numbers up to a given limit."}
        ]
    }
]

# Note: For real fine-tuning, you need 50-100+ examples minimum!
# This is just demonstrating the format.

print(f"✅ Created {len(training_data)} training examples")
print(f"   (Real fine-tuning needs 50-100+ examples)")

## 🧪 Step 3: Validate & Save as JSONL

In [ ]:
# Validate the format
def validate_training_data(data):
    errors = []
    for i, example in enumerate(data):
        if "messages" not in example:
            errors.append(f"Example {i}: Missing 'messages' key")
            continue
        messages = example["messages"]
        if not any(m["role"] == "assistant" for m in messages):
            errors.append(f"Example {i}: No assistant message")
        for msg in messages:
            if msg["role"] not in ["system", "user", "assistant"]:
                errors.append(f"Example {i}: Invalid role '{msg['role']}'")
    return errors

errors = validate_training_data(training_data)
if errors:
    print("❌ Validation errors:")
    for e in errors:
        print(f"  {e}")
else:
    print("✅ All training examples valid!")

# Save as JSONL
output_file = "training_data.jsonl"
with open(output_file, "w") as f:
    for example in training_data:
        f.write(json.dumps(example) + "\n")

print(f"✅ Saved to {output_file}")

## 🧪 Step 4: Upload File & Create Fine-Tuning Job

⚠️ **Note**: The cells below show the exact API calls. Uncomment to run (costs ~$0.50-$5 depending on dataset size).

In [ ]:
# # Upload training file
# file_response = client.files.create(
#     file=open("training_data.jsonl", "rb"),
#     purpose="fine-tune"
# )
# file_id = file_response.id
# print(f"✅ File uploaded: {file_id}")

# # Create fine-tuning job
# job = client.fine_tuning.jobs.create(
#     training_file=file_id,
#     model="gpt-4o-mini-2024-07-18",
#     hyperparameters={
#         "n_epochs": 3,  # Number of training passes
#     }
# )
# print(f"✅ Fine-tuning job created: {job.id}")
# print(f"   Status: {job.status}")

print("ℹ️ Code above is commented out to avoid charges.")
print("   Uncomment to run actual fine-tuning (~$0.50 for small datasets).")

In [ ]:
# # Monitor the job
# job_status = client.fine_tuning.jobs.retrieve(job.id)
# print(f"Status: {job_status.status}")
# print(f"Fine-tuned model: {job_status.fine_tuned_model}")

# # List events
# events = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=5)
# for event in events.data:
#     print(f"  {event.created_at}: {event.message}")

print("ℹ️ After job completes (~5-15 min), use the fine-tuned model:")
print('   model="ft:gpt-4o-mini-2024-07-18:org:custom-name:id"')

## 🧪 Step 5: Use the Fine-Tuned Model

In [ ]:
# # Using the fine-tuned model (once ready)
# response = client.chat.completions.create(
#     model="ft:gpt-4o-mini-2024-07-18:your-org::job-id",  # Your fine-tuned model ID
#     messages=[
#         {"role": "system", "content": "You are a coding tutor."},
#         {"role": "user", "content": "What are Python context managers?"}
#     ]
# )
# print(response.choices[0].message.content)

# The response should follow the pattern:
# 1. Concept explanation
# 2. Code example
# 3. Practice exercise
# ...because that's what we trained it to do!

print("ℹ️ The fine-tuned model will automatically follow the trained pattern:")
print("   Concept → Code Example → Practice Exercise")

## 🧪 Step 6: Cost Estimation

In [ ]:
import tiktoken

def estimate_fine_tuning_cost(data, model="gpt-4o-mini-2024-07-18", epochs=3):
    """Estimate OpenAI fine-tuning cost."""
    enc = tiktoken.encoding_for_model("gpt-4o-mini")
    total_tokens = 0
    for example in data:
        for msg in example["messages"]:
            total_tokens += len(enc.encode(msg["content"]))
    
    training_tokens = total_tokens * epochs
    # gpt-4o-mini fine-tuning: ~$0.0008 per 1K training tokens (as of 2024)
    cost = (training_tokens / 1000) * 0.0008
    
    print(f"📊 Cost Estimation:")
    print(f"   Total tokens per example: ~{total_tokens // len(data)}")
    print(f"   Total training tokens: {training_tokens:,} ({epochs} epochs)")
    print(f"   Estimated cost: ${cost:.4f}")
    return cost

estimate_fine_tuning_cost(training_data)

## ✍️ Practice Challenge

Create a training dataset (10+ examples) to fine-tune a model that:
- Always responds with a structured JSON output
- Categorizes customer support messages into: billing, technical, general
- Includes a confidence score and suggested action

<details>
<summary>💡 Hint</summary>

```python
example = {
    "messages": [
        {"role": "system", "content": "Classify support tickets. Respond in JSON."},
        {"role": "user", "content": "My payment failed"},
        {"role": "assistant", "content": '{"category": "billing", "confidence": 0.95, "action": "escalate_to_billing"}'}
    ]
}
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Training data format: JSONL with system/user/assistant msgs.   │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Minimum: 50-100 examples for meaningful fine-tuning.           │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Pipeline: Prepare data → Upload → Create job → Monitor → Use. │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Cost: Very low for gpt-4o-mini (~$0.50 for small datasets).   │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Always validate data format before uploading!                  │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: MCP Protocol — Model Context Protocol!

---

### 🎉 You Know How to Fine-Tune!
From data prep to deployment — the complete fine-tuning pipeline. 🚀